# Stress Prediction V33 — Pair Quantile 48% / Weight 20%

28개 행 단위 변수쌍 이웃 예측의 분위수를 48%로 조정하고 비중을 20%로 적용합니다. 5-Fold MAE와 제출 파일을 함께 생성합니다.


In [1]:
from __future__ import annotations

import argparse
import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.52
PAIR_WEIGHT = 0.15

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant", fill_value="missing"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_quantile_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_test = add_feature_copies(test_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    random_state=RANDOM_SEED,
                    n_jobs=1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    matrix = pipeline.named_steps["preprocessor"].transform(weighted_test)
    forest = pipeline.named_steps["model"]
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in forest.estimators_]
    )
    return np.clip(
        np.round(np.quantile(tree_predictions, TREE_QUANTILE, axis=1), 2),
        0.0,
        1.0,
    )


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Train 분포만으로 Train과 한 개씩 독립적인 Test 값을 변환."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(
        sorted_train, train_values, side="right"
    ) / len(train_values)
    test_rank = np.searchsorted(
        sorted_train, test_values, side="right"
    ) / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """각 2차원 피처쌍에서 가장 가까운 Train target을 분위수 집계."""
    train_rank = {}
    test_rank = {}
    for column in PAIR_COLUMNS:
        train_rank[column], test_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            test_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(test_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(test_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_index = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_index])
    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def validate_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")

In [2]:
from sklearn.model_selection import KFold

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
validate_inputs(train, test, submission)
X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

V33_PAIR_QUANTILE = 0.48
V33_PAIR_WEIGHT = 0.20
PAIR_NAMES = list(combinations(PAIR_COLUMNS, 2))


def pair_prediction_v33(train_features, valid_features, target):
    train_rank, valid_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], valid_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(float),
            valid_features[column].to_numpy(float),
        )
    target_values = target.to_numpy(float)
    predictions = []
    for first, second in PAIR_NAMES:
        distance = (
            np.abs(valid_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(valid_rank[second][:, None] - train_rank[second][None, :])
        )
        predictions.append(target_values[np.argmin(distance, axis=1)])
    return np.quantile(
        np.column_stack(predictions), V33_PAIR_QUANTILE, axis=1
    )


def predict_v33(train_features, valid_features, target):
    tree_prediction = tree_quantile_prediction(
        train_features, valid_features, target
    )
    pair_prediction = pair_prediction_v33(
        train_features, valid_features, target
    )
    return np.clip(
        np.round(
            (1.0 - V33_PAIR_WEIGHT) * tree_prediction
            + V33_PAIR_WEIGHT * pair_prediction,
            2,
        ),
        0.0,
        1.0,
    )


cv = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(y), dtype=float)
fold_rows = []
for fold, (train_idx, valid_idx) in enumerate(cv.split(X), 1):
    oof[valid_idx] = predict_v33(
        X.iloc[train_idx], X.iloc[valid_idx], y.iloc[train_idx]
    )
    score = mean_absolute_error(y.iloc[valid_idx], oof[valid_idx])
    fold_rows.append({"Fold": fold, "V33_MAE": score})
    print(f"Fold {fold} V33 MAE: {score:.6f}")

cv_mae = mean_absolute_error(y, oof)
print("V33 CV MAE:", f"{cv_mae:.6f}")
display(pd.DataFrame(fold_rows))

test_prediction = predict_v33(X, X_test, y)
output = submission.copy()
output[TARGET] = test_prediction
output_path = OUTPUT_DIR / "submit_v33_pair_q48_weight20.csv"
output.to_csv(output_path, index=False, encoding="utf-8")
print("Saved:", output_path)
print("Prediction range:", test_prediction.min(), test_prediction.max())
display(output.head(10))



Fold 1 V33 MAE: 0.134233
Fold 2 V33 MAE: 0.149700
Fold 3 V33 MAE: 0.147767
Fold 4 V33 MAE: 0.159083
Fold 5 V33 MAE: 0.150267
V33 CV MAE: 0.148210


,Fold,V33_MAE
0,1,0.134233
1,2,0.149700
2,3,0.147767
3,4,0.159083
4,5,0.150267


Saved: /Users/bitsaem/Desktop/stress_project/submit_v33_pair_q48_weight20.csv
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.56
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.49
4,TEST_0004,0.53
5,TEST_0005,0.45
6,TEST_0006,0.78
7,TEST_0007,0.47
8,TEST_0008,0.57
9,TEST_0009,0.88
